# 输入层
sentence -> tokenizor -> padding -> embedding -> add position embedding
-> masked attention

In [13]:
import tiktoken
import torch 
import torch.nn as nn
import numpy as np 
from torch.utils.data import  Dataset, DataLoader
encoding = tiktoken.get_encoding("gpt2")

In [14]:
batch = 2
max_len = 16
vocab_size = encoding.n_vocab
data = ["你好！","回家了吧。"]
print(encoding.encode(data[0]))
print(encoding.encode(data[1]))
x = [encoding.encode(sentence) for sentence in data ]
valid_length = [len(sentence) for sentence in x]
y = [sentence[1:] + [-100]*(max_len+1-len(sentence)) if max_len-len(sentence)>0 else sentence[1:]  for sentence in x ]
x = [sentence + [encoding.eot_token]*(max_len-len(sentence)) if max_len-len(sentence)>0 else sentence for sentence in x ]
x= torch.tensor(x)
y = torch.tensor(y)
pad_mask = torch.ones(batch,max_len,max_len)==0
for i,length in enumerate(valid_length):
    pad_mask[i,:,length:] = True

[19526, 254, 25001, 121, 171, 120, 223]
[32368, 252, 22522, 114, 12859, 228, 28938, 100, 16764]


In [15]:
y

tensor([[  254, 25001,   121,   171,   120,   223,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100],
        [  252, 22522,   114, 12859,   228, 28938,   100, 16764,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100]])

In [16]:
word_embedding = torch.nn.Embedding(encoding.n_vocab,256)
pos_embedding = torch.nn.Embedding(max_len,256)
x = torch.tensor(x)
embedding = word_embedding(x) + pos_embedding(torch.arange(max_len)).unsqueeze(0)

C:\Users\24237\AppData\Local\Temp\ipykernel_13452\1588742831.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x)


In [17]:
class MultiHeadAttention(nn.Module):
    def __init__(self,hidden_dim = 256,head_num = 8,max_len = 16):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.head_num = head_num
        self.max_len = max_len
        self.q_layer = nn.Linear(hidden_dim,hidden_dim,bias=False)
        self.k_layer = nn.Linear(hidden_dim,hidden_dim,bias=False)
        self.v_layer = nn.Linear(hidden_dim,hidden_dim,bias=False)
        self.ff = nn.Linear(hidden_dim,hidden_dim)
        self.norm_layer = nn.LayerNorm(hidden_dim)

        
    def forward(self,x):
        x,pad_mask = x
        q= self.q_layer(x).view(x.shape[0],x.shape[1],self.head_num,self.hidden_dim//self.head_num).transpose(1,2)
        k = self.k_layer(x).view(x.shape[0],x.shape[1],self.head_num,self.hidden_dim//self.head_num).transpose(1,2)
        v = self.v_layer(x).view(x.shape[0],x.shape[1],self.head_num,self.hidden_dim//self.head_num).transpose(1,2)
        attention_mask = torch.tril(torch.ones(x.shape[0],self.max_len,self.max_len))==0
        attention_score = torch.softmax(torch.masked_fill(torch.masked_fill(q@k.transpose(2,3),pad_mask.unsqueeze(1),-torch.inf),attention_mask.unsqueeze(1),-torch.inf),-1)
        attention_x = attention_score @v
        attention_x = attention_x.transpose(1,2).reshape((x.shape[0],self.max_len,self.hidden_dim))
        res_x = self.norm_layer(attention_x) + x
        ff_1 = self.ff(res_x)
        ff_2 = self.norm_layer(ff_1) + ff_1
        return (ff_2,pad_mask)
class TransformerDecoder(nn.Module):
    def __init__(self, hidden_dim = 256,head_num = 8,layers = 8,vocab_size =10000,max_len = 16,*args, **kwargs):
        super().__init__(*args, **kwargs)
        self.word_embedding = torch.nn.Embedding(vocab_size,hidden_dim)
        self.pos_embedding = torch.nn.Embedding(max_len,hidden_dim)
        self.attention_layers = nn.Sequential(*[MultiHeadAttention(hidden_dim,head_num,max_len)]*layers)
    def forward(self,x,pad_mask):
        embedding = self.word_embedding(x) + self.pos_embedding(torch.arange(x.shape[1])).unsqueeze(0)
        out,_ = self.attention_layers((embedding,pad_mask))
        return out
class LLM(nn.Module):
     def __init__(self, hidden_dim = 256,head_num = 8,layers = 8,vocab_size =10000,max_len = 16):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.decoder = TransformerDecoder(hidden_dim ,head_num ,layers ,vocab_size,max_len)
        self.llm_head = nn.Sequential(nn.Linear(hidden_dim,vocab_size),nn.Softmax(dim=-1))

     def forward(self,x,y,pad_mask):
         decoder_out = self.decoder(x,pad_mask)
         out = self.llm_head(decoder_out)
         loss = nn.functional.cross_entropy(out.view(-1,self.vocab_size),y.view(-1))
         return loss
     def generate(self,x,max_new_tokens = 16):
         valid_length = len(x)
         x = x + [encoding.eot_token]*(self.max_len-len(x)) if self.max_len-len(x)>0 else x
         x= torch.tensor(x).view(1,-1)
         pad_mask = torch.ones(1,self.max_len,self.max_len)==0
         generate_text = []
         for i in range(max_new_tokens):
            pad_mask[:,:,valid_length:] = True
            decoder_out = self.decoder(x,pad_mask)
            out = self.llm_head(decoder_out)
            next_token = torch.argmax(out,dim=-1)[0,valid_length-1]
            generate_text.append(next_token.item())
            if i+1!=max_new_tokens:
                x[0,i+1] = next_token
                valid_length+=1

         return generate_text
         
            

In [18]:

class MyDataset(Dataset):
    def __init__(self,batch_size=4, max_len = 16):
        self.pad_mask = torch.ones(max_len,max_len)==0
        self.x = []
        self.y = []
        with open("ch02\\01_main-chapter-code\\the-verdict.txt","r") as f:
            text = "".join(f.readlines())
            text_ids = encoding.encode(text)
            for i in range(0,len(text_ids)-max_len-1,5):
                self.x.append(text_ids[i:i+max_len])
                self.y.append(text_ids[i+1:i+max_len+1])

    def __len__(self):
        return len(self.x)

    def __getitem__(self, key):
        return torch.tensor(self.x[key]), torch.tensor(self.y[key]),    torch.tensor(self.pad_mask)


In [ ]:
import torch.optim as optim
dataset = MyDataset(batch_size=4,max_len=256)
dataLoader = DataLoader(dataset,4)
model = LLM(vocab_size=vocab_size,max_len=256)
optimizer = optim.Adam(model.parameters(), lr=0.00001)
step = 0
for i in range(100):
    for x,y,pad_mask in dataLoader:
        optimizer.zero_grad()
        loss  = model(x,y,pad_mask)
        loss.backward()
        optimizer.step()
        step +=1
        print(f"Epoch：{i+1},step:{step},loss:{loss.item()}")



C:\Users\24237\AppData\Local\Temp\ipykernel_13452\3357263015.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(self.x[key]), torch.tensor(self.y[key]),    torch.tensor(self.pad_mask)


Epoch：1,step:1,loss:10.824902534484863
Epoch：1,step:2,loss:10.824905395507812
Epoch：1,step:3,loss:10.824905395507812
Epoch：1,step:4,loss:10.824905395507812
Epoch：1,step:5,loss:10.824908256530762
Epoch：1,step:6,loss:10.824904441833496
Epoch：1,step:7,loss:10.824902534484863
Epoch：1,step:8,loss:10.82490348815918
Epoch：1,step:9,loss:10.824905395507812
Epoch：1,step:10,loss:10.824908256530762
Epoch：1,step:11,loss:10.824901580810547
Epoch：1,step:12,loss:10.824901580810547
Epoch：1,step:13,loss:10.824905395507812
Epoch：1,step:14,loss:10.824906349182129
Epoch：1,step:15,loss:10.82490348815918
Epoch：1,step:16,loss:10.82490062713623
Epoch：1,step:17,loss:10.824904441833496
Epoch：1,step:18,loss:10.824905395507812
Epoch：1,step:19,loss:10.824906349182129
Epoch：1,step:20,loss:10.824906349182129
Epoch：1,step:21,loss:10.824907302856445
Epoch：1,step:22,loss:10.824902534484863
Epoch：1,step:23,loss:10.824904441833496
Epoch：1,step:24,loss:10.824904441833496
Epoch：1,step:25,loss:10.824904441833496
Epoch：1,step

In [ ]:
s[2].shape

torch.Size([4, 16, 16])

([40,
  367,
  2885,
  1464,
  1807,
  3619,
  402,
  271,
  10899,
  2138,
  257,
  7026,
  15632,
  438,
  2016,
  257],
 [367,
  2885,
  1464,
  1807,
  3619,
  402,
  271,
  10899,
  2138,
  257,
  7026,
  15632,
  438,
  2016,
  257,
  922],
 tensor([[False, False, False, False, False, False, False, False, False, False,
          False, False, False, False, False, False],
         [False, False, False, False, False, False, False, False, False, False,
          False, False, False, False, False, False],
         [False, False, False, False, False, False, False, False, False, False,
          False, False, False, False, False, False],
         [False, False, False, False, False, False, False, False, False, False,
          False, False, False, False, False, False],
         [False, False, False, False, False, False, False, False, False, False,
          False, False, False, False, False, False],
         [False, False, False, False, False, False, False, False, False, False,
        